# Used Phone Price Predictor — EDA & Modelling Walkthrough

**Author:** ML Engineering Project  
**Dataset:** Synthetic — generated by `src/data_generator.py`  
**Scope:** India second-hand smartphone market (2019–2025)

> ⚠️ **Synthetic data disclaimer:** All 6 000 rows are programmatically generated.
> The pricing formula encodes realistic assumptions (exponential depreciation,
> brand-specific retention rates, battery health penalty, condition multipliers)
> but does **not** represent real transaction data. The exercise is designed to
> practise the full ML pipeline — not to deploy a production price engine.

---

## Notebook outline

1. Environment setup
2. Data loading & basic EDA
3. Data cleaning & preprocessing
4. Univariate analysis
5. Bivariate analysis
6. Multivariate analysis
7. Feature engineering
8. Model training & comparison
9. Best model evaluation
10. Conclusions


---
## 1. Environment setup

In [ ]:
import sys
from pathlib import Path

# Add src/ to path so we can import project modules
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

from config import (
    ALL_FEATURES, NUMERIC_FEATURES, ORDINAL_FEATURES, NOMINAL_FEATURES,
    TARGET, RANDOM_SEED, CURRENT_YEAR, RAW_DATA_PATH,
    TRAIN_PATH, TEST_PATH, MODEL_PATH, METRICS_PATH, FIGURES_DIR,
    SCREEN_CONDITION_ORDER, BODY_CONDITION_ORDER,
)

# Consistent plot style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

print('Setup complete. ROOT:', ROOT)

---
## 2. Data loading & basic EDA

We load the raw CSV (generated by `data_generator.py`).  
The raw file is **intentionally messy**: missing values, duplicates, outliers, and inconsistent casing.

In [ ]:
# Generate data if it doesn't exist yet
if not RAW_DATA_PATH.exists():
    from data_generator import generate
    generate()

df_raw = pd.read_csv(RAW_DATA_PATH)
print(f'Shape: {df_raw.shape}')
df_raw.head()

In [ ]:
df_raw.tail()

In [ ]:
df_raw.info()

In [ ]:
df_raw.dtypes

In [ ]:
df_raw.describe(include='all').T

In [ ]:
# Missing value report
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
pd.DataFrame({'Missing': missing, 'Missing %': missing_pct}).query('Missing > 0')

In [ ]:
# Duplicate rows
print(f'Duplicate rows: {df_raw.duplicated().sum()}')

In [ ]:
# Inconsistent casing (brand column sample)
df_raw['brand'].value_counts().head(20)

**Observations:**
- ~3 % missing values in `battery_health_pct`, `storage_gb`, `body_condition` (injected deliberately).
- ~90 duplicate rows (1.5 % of 6 000).
- `brand` and condition columns have mixed casing/whitespace — need standardisation.
- Some `battery_health_pct` values exceed 100 (physically impossible) — outliers injected for practice.

---
## 3. Data cleaning & preprocessing

**Strategy:**
1. Strip / title-case all string columns.
2. Drop exact duplicate rows.
3. Cap `battery_health_pct` at 100 (physically impossible > 100).
4. Remove `price_inr` outliers using IQR with k = 3 (very permissive; only removes egregious injected outliers).
5. All imputation is done **inside the sklearn Pipeline** on training data only — no leakage.

In [ ]:
from preprocessing import clean_strings, remove_outliers, load_and_prepare, split_and_save

df = load_and_prepare(RAW_DATA_PATH)
print(f'Clean shape: {df.shape}')
df.head()

In [ ]:
# Verify: no battery health > 100 after cleaning
print('Max battery_health_pct:', df['battery_health_pct'].max())
# Verify: brand is clean
print('Unique brands:', sorted(df['brand'].dropna().unique()))

In [ ]:
# Missing values remaining (will be handled inside sklearn pipeline)
df.isnull().sum()[df.isnull().sum() > 0]

In [ ]:
# Train/test split
train_df, test_df = split_and_save(df)
X_train = train_df[ALL_FEATURES]
y_train = train_df[TARGET].values
X_test  = test_df[ALL_FEATURES]
y_test  = test_df[TARGET].values
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

---
## 4. Univariate analysis

We look at individual distributions to understand the shape of each feature.

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df[TARGET] / 1000, bins=50, color='#2d6a6a', edgecolor='white')
axes[0].set_xlabel('Price (₹ thousands)')
axes[0].set_title('Distribution of resale price (raw)')

axes[1].hist(np.log1p(df[TARGET]), bins=50, color='#4a9b9b', edgecolor='white')
axes[1].set_xlabel('log1p(price)')
axes[1].set_title('Distribution of resale price (log-transformed)')

plt.tight_layout()
plt.show()
print('The log-transformed distribution is closer to normal — justifies TransformedTargetRegressor.')

In [ ]:
# Numeric features — histograms
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(NUMERIC_FEATURES):
    axes[i].hist(df[col].dropna(), bins=30, color='#2d6a6a', edgecolor='white')
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')

# Hide unused subplot
if len(NUMERIC_FEATURES) < len(axes):
    for j in range(len(NUMERIC_FEATURES), len(axes)):
        axes[j].set_visible(False)

plt.suptitle('Numeric feature distributions', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical features — count plots
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, col in zip(axes, ['brand', 'screen_condition', 'body_condition', 'has_box_bill']):
    counts = df[col].value_counts()
    ax.bar(counts.index.astype(str), counts.values, color='#2d6a6a', edgecolor='white')
    ax.set_title(col, fontsize=10)
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

---
## 5. Bivariate analysis

We explore how each feature relates to the target (`price_inr`).

In [ ]:
# Price vs age
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(df['age_years'], df[TARGET] / 1000, alpha=0.3, s=8, color='#2d6a6a')
ax.set_xlabel('Age (years)')
ax.set_ylabel('Resale price (₹ thousands)')
ax.set_title('Price vs phone age')
plt.tight_layout()
plt.show()

In [ ]:
# Price vs battery health
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(df['battery_health_pct'], df[TARGET] / 1000, alpha=0.3, s=8, color='#4a9b9b')
ax.set_xlabel('Battery health (%)')
ax.set_ylabel('Resale price (₹ thousands)')
ax.set_title('Price vs battery health (note: nonlinear relationship)')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot: price by screen condition (ordered)
fig, ax = plt.subplots(figsize=(7, 4))
order = SCREEN_CONDITION_ORDER
data_for_box = [df[df['screen_condition'] == sc][TARGET].dropna() / 1000 for sc in order]
bp = ax.boxplot(data_for_box, labels=order, patch_artist=True, notch=False)
colors = ['#cc4444', '#e8a830', '#2d6a6a']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_xlabel('Screen condition')
ax.set_ylabel('Price (₹ thousands)')
ax.set_title('Resale price by screen condition')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot: median resale price by brand
brand_median = df.groupby('brand')[TARGET].median().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(brand_median.index, brand_median.values / 1000, color='#2d6a6a', edgecolor='white')
ax.set_ylabel('Median resale price (₹ thousands)')
ax.set_title('Median resale price by brand')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

---
## 6. Multivariate analysis

Correlation heatmap helps identify collinear features and the strength of each
feature's linear relationship with the target.

In [ ]:
# Correlation heatmap (numeric features + target)
num_cols = NUMERIC_FEATURES + [TARGET]
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Pearson correlation — numeric features + target', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot on a small representative subset
subset_cols = ['age_years', 'battery_health_pct', 'original_price_inr', TARGET]
sample = df[subset_cols].dropna().sample(500, random_state=RANDOM_SEED)
sample[TARGET] = sample[TARGET] / 1000  # scale for readability
sample.rename(columns={TARGET: 'price_k'}, inplace=True)

g = sns.pairplot(sample, diag_kind='kde', plot_kws={'alpha': 0.3, 's': 10},
                 diag_kws={'color': '#2d6a6a'},
                 height=2.2)
g.fig.suptitle('Pairplot: key numeric features vs price (500-row sample)', y=1.02, fontsize=11)
plt.show()

**Observations:**
- `original_price_inr` has the strongest positive correlation with `price_inr`.
- `age_years` has the strongest negative correlation.
- `battery_health_pct` shows a nonlinear relationship (barely visible in linear Pearson r).
- No severe multicollinearity between independent features.

---
## 7. Feature engineering

- `age_years = CURRENT_YEAR − release_year` — captures depreciation effect.  
- `original_price_inr` — the launch price is a strong anchor for resale value.  
- **Not used:** `price_retention_ratio = price_inr / original_price_inr` — this is derived from the target; using it would be **data leakage**.

The sklearn `ColumnTransformer` pipeline (built in `preprocessing.py`) applies:
| Column group | Imputer | Encoder / Scaler |
|---|---|---|
| Numeric (7 cols) | Median | StandardScaler |
| Ordinal (2 cols) | Most frequent | OrdinalEncoder (explicit order) |
| Nominal (2 cols) | Most frequent | OneHotEncoder (handle_unknown=ignore) |

In [ ]:
from preprocessing import build_feature_pipeline
from sklearn.pipeline import Pipeline
import numpy as np

# Demonstrate the pipeline on a small sample
pipe = build_feature_pipeline()
pipe.fit(X_train)
X_transformed = pipe.transform(X_train)
print(f'Input shape:  {X_train.shape}')
print(f'Output shape: {X_transformed.shape}  (OHE expands brand/model to many columns)')

---
## 8. Model training & comparison

We use `TransformedTargetRegressor` so training happens on `log1p(price_inr)` and
all predictions are automatically inverse-transformed (`expm1`) to INR.

All models share the **same** `ColumnTransformer` preprocessor → no leakage, no mismatch.

In [ ]:
# ── Run full training pipeline ──
# (This calls the same code as `python src/train.py`)
import importlib, train as tr

tr.main()

In [ ]:
# Load saved metrics and display comparison table
from evaluate import load_metrics

metrics = load_metrics()
best_model_name = metrics['best_model']
tm = metrics['test_metrics']

df_metrics = pd.DataFrame(tm).T.sort_values('RMSE')
df_metrics.index.name = 'Model'
display(df_metrics.style.format({'MAE': '{:,.0f}', 'RMSE': '{:,.0f}',
                                  'MSE': '{:,.0f}', 'R2': '{:.4f}',
                                  'MAPE': '{:.1f}'}).highlight_min(subset=['RMSE'], color='#d4edea'))
print(f'\nBest model: {best_model_name}')

In [ ]:
# Show saved figures
from IPython.display import Image, display as ipy_display

for fname in ['model_comparison.png', 'knn_k_vs_rmse.png']:
    p = FIGURES_DIR / fname
    if p.exists():
        ipy_display(Image(str(p)))

In [ ]:
# Decision tree visualisation (shallow, max_depth=4)
dt_fig = FIGURES_DIR / 'decision_tree.png'
if dt_fig.exists():
    ipy_display(Image(str(dt_fig), width=900))

---
## 9. Best model evaluation

We load the saved best model and evaluate it on the held-out test set.

In [ ]:
from predict import load_model, predict_price
from evaluate import compute_metrics

model = load_model()

y_pred = model.predict(X_test)
m = compute_metrics(y_test, y_pred)

print(f'Test-set metrics — {best_model_name}')
for k, v in m.items():
    unit = ' INR' if k in ('MAE', 'RMSE') else (' %' if k == 'MAPE' else '')
    print(f'  {k:<6} = {v:>12,.2f}{unit}')

In [ ]:
# Actual vs predicted
for fname in ['actual_vs_predicted.png', 'residuals.png', 'feature_importances.png']:
    p = FIGURES_DIR / fname
    if p.exists():
        ipy_display(Image(str(p)))

In [ ]:
# Demonstrate single prediction via predict.py
spec = {
    'brand':                'Apple',
    'model':                'iPhone 13',
    'release_year':         2021,
    'age_years':            3.0,
    'storage_gb':           128,
    'ram_gb':               4,
    'battery_health_pct':   87.0,
    'screen_condition':     'minor_scratches',
    'body_condition':       'good',
    'has_box_bill':         1,
    'warranty_months_left': 0,
    'original_price_inr':   79900,
    'market_demand_index':  7.5,
}
price = predict_price(model, spec)
print(f'Predicted resale price: ₹{price:,}')

---
## 10. Conclusions

### Which model won and why?

**Random Forest** achieved the best test-set RMSE and R² across all five model families.

| Why RF wins on this task |
|---|
| Captures the **nonlinear** battery health cliff (below 80%) that linear models miss |
| Handles **interactions** between brand × age × condition without explicit feature crosses |
| **Robust** to the remaining noise injected by the log-normal price variance |
| No need for feature scaling (unlike SVR and KNN) |
| 200–300 trees provide **ensemble averaging** that reduces variance vs a single tree |

### Model ranking (approximate)

1. **Random Forest** — best RMSE / R²
2. **Decision Tree** (tuned) — close second; interpretable
3. **SVR** — strong but sensitive to scaling and kernel choice
4. **KNN** — decent; hurt by the high-dimensional OHE space
5. **Linear Regression** — underfit; misses nonlinear interactions

### Limitations

- The dataset is **synthetic** — model performance on real data will differ.
- Only 30 phone models. Unseen models at inference fall back to the OHE 'ignore' strategy.
- Market prices drift over time; the model needs retraining for production use.
- `market_demand_index` is a constructed proxy, not a real signal.

---
*End of notebook. Run `streamlit run app.py` from the project root to launch the interactive app.*